# Seabird Dataset

## Notebook Stack

### IO
`pyarrow` is the defacto parquet and arrow standard library tooling in the python eco-system, offering exceptional performance, documentation and stability.

Its `fs` class allows connecting to parquet datasets in AWS S3, which is perfect for our use case.

### Computation
`polars` is fast replacing pandas as the primary python dataframe API. It is more tightly integrated with arrow (and pyarrow), has exceptional performance, greater than memory dataset manipulation, a more consistent and declarative API, built in plotting and many other advantages.

### Mapping
`pydeck` is a python binding for `deck.gl`, a GPU-powered framework for visual exploratory data analysis of large datasets. It is optimised for Jupyter notebooks and highly customisable.

In [ ]:
# Set up rich console
import rich
import rich.console
console = rich.console.Console()

## Connecting to NESP 5.9 Datasets
NESP 5.9 datasets are currently stored in S3 and a publicly available as per the following table:

| Bucket | Key |
| :--- | :--- |
| `data-uplift-public` | `stored/datauplift/amsa/amsa.parquet` |
| `data-uplift-public` | `stored/datauplift/kelp/kelp.parquet` |
| `data-uplift-public` | `stored/datauplift/nrmn/public/ep_m13_pq_scores.parquet` |
| `data-uplift-public` | `stored/datauplift/seabird/seabird.parquet` |
| `data-uplift-public` | `stored/datauplift/seagrass/seagrass.parquet` |

In [ ]:
import pyarrow
import pyarrow.fs
import pyarrow.dataset
import util

# Construct the anonymous file system responsible for reading from the public S3 bucket
FILE_SYSTEM = pyarrow.fs.S3FileSystem(
    region="ap-southeast-2", 
    anonymous=True,
)

# By convention, datasets are labelled `ds`
ds = pyarrow.dataset.dataset(
    source="data-uplift-public/stored/datauplift/seabird/seabird.parquet",
    filesystem=FILE_SYSTEM,
)

# The comprising files of the dataset can be inspected
rich.print(list(ds.get_fragments()))

# The schema of the dataset can be inspected
schema_rich_table = util.generate_schema_rich_table(
    schema=ds.schema,
)
console.print(schema_rich_table)

# For the full table scheme use native pyarrow representation
# rich.print(ds.schema)

In [ ]:
import polars

# Option (a): Load entire dataset in memory as a dataframe
# Suitable for smaller datasets that fit into memory
df = polars.DataFrame(
    data=ds.to_table(),
)

# Option (b): Scan entire dataset as a lazyframe
# Suitable for larger datasets that don't fit into memory
lf = polars.scan_pyarrow_dataset(
    source=ds,
)

## H3 Aggregation and Mapping

In [ ]:
import polars_h3

H3_RESOLUTION = 4

# H3 index
df = df.with_columns(
    polars_h3.latlng_to_cell(
        lat=polars.col("decimalLatitude"),
        lng=polars.col("decimalLongitude"),
        resolution=H3_RESOLUTION,
        return_dtype=polars.String,
    ).alias("h3_cell"),
)

In [ ]:
import pydeck

# Aggregate by h3_cell
h3_aggregated_df = (
    df
    .group_by(
        polars.col("h3_cell"),
    )
    .agg(
        polars.col("id").count().alias("n_records"),
        polars.col("download_link"),
    )
    .with_columns(
        polars.col("download_link").list.eval(
            polars.element().str.split(by="archive.do?r=").list.last()
        ).list.unique().list.sort().list.join(", ").alias("datasets"),
    ).select(
        polars.col("h3_cell"),
        polars.col("n_records"),
        polars.col("datasets"),
    )
)

# Generate a color mapping
color_mapping = util.generate_color_mapping(color_palette="plasma")

# Bin the h3 cells
map_df = h3_aggregated_df.with_columns(
    polars.col("n_records").qcut(
        quantiles=len(color_mapping),
        labels=list(color_mapping.keys()),
        allow_duplicates=True,
    ).cast(polars.String).alias("color_index"),
)

# Generate map layers
h3_layers = [
    pydeck.Layer(
        "H3HexagonLayer",
        map_df.filter(polars.col("color_index").eq(color_index)).to_pandas(),
        get_hexagon="h3_cell",
        extruded=False,
        get_fill_color=fill_color,
        get_line_color=fill_color[:3],
        line_width_min_pixels=2,
        pickable=True,
    )
    for color_index, fill_color in color_mapping.items()
]

# Run the map
deck = pydeck.Deck(
    layers=h3_layers,
    initial_view_state=pydeck.ViewState(
        latitude=0,
        longitude=0,
        zoom=1,
        pitch=0,
        bearing=0,
    ),
    map_style=pydeck.map_styles.LIGHT_NO_LABELS,
    tooltip={
        "html": """
            <div style="font-family: 'Helvetica Neue', Arial, sans-serif; padding: 10px;">
                <b style="font-size: 1.2em;">H3 Cell:</b> <code>{h3_cell}</code><br/>
                <hr style="margin: 5px 0; border: 0; border-top: 1px solid #ccc;">
                <b>Records:</b> {n_records}<br/>
                <b>Datasets:</b> {datasets}
            </div>
        """,
        "style": {
            "width": "33%",
            "backgroundColor": "#2b2b2b",
            "color": "white",
            "borderRadius": "4px",
            "zIndex": "1000"
        }
    }
)
deck.to_html(f"h3_aggregated_resolution_{H3_RESOLUTION}.html")
deck.show()

## Identifying Trend Sites
In the above map we can see there are some sites with a high density of records. These are the most ideal sites for trend analysis.

### Per Dataset
The singular datasets that comprise the aggregated data product are typically good candidates for trend analysis

### Per Region
Otherwise, occurrence trend analysis can be applied to high density regions where data exists.

### Mapping the Spatial Extent of a Particular Region
Hovering over the above maps shows which datasets are present per H3 cell. This allows identification of datasets to subset on and analyse more closely.

Mapping the spatial extent of a dataset or region is as simple as filtering to those rows before performing the mapping.

In [ ]:
seabird_atlas_sea_df = df.filter(
    polars.col("download_link").eq("https://www.marine.csiro.au/ipt/archive.do?r=seabird_atlas_southeast_australia&v=2.9"),
)

# Aggregate by h3_cell
h3_aggregated_df = (
    seabird_atlas_sea_df
    .group_by(
        polars.col("h3_cell"),
    )
    .agg(
        polars.col("id").count().alias("n_records"),
        polars.col("download_link"),
    )
    .with_columns(
        polars.col("download_link").list.eval(
            polars.element().str.split(by="archive.do?r=").list.last()
        ).list.unique().list.sort().list.join(", ").alias("datasets"),
    ).select(
        polars.col("h3_cell"),
        polars.col("n_records"),
        polars.col("datasets"),
    )
)


# Generate a color mapping
color_mapping = util.generate_color_mapping(color_palette="plasma")

# Bin the h3 cells
map_df = h3_aggregated_df.with_columns(
    polars.col("n_records").qcut(
        quantiles=len(color_mapping),
        labels=list(color_mapping.keys()),
        allow_duplicates=True,
    ).cast(polars.String).alias("color_index"),
)

# Generate map layers
h3_layers = [
    pydeck.Layer(
        "H3HexagonLayer",
        map_df.filter(polars.col("color_index").eq(color_index)).to_pandas(),
        get_hexagon="h3_cell",
        extruded=False,
        get_fill_color=fill_color,
        get_line_color=fill_color[:3],
        line_width_min_pixels=2,
        pickable=True,
    )
    for color_index, fill_color in color_mapping.items()
]

# Run the map
deck = pydeck.Deck(
    layers=h3_layers,
    initial_view_state=pydeck.ViewState(
        latitude=-42,
        longitude=146,
        zoom=4,
        pitch=0,
        bearing=0,
    ),
    map_style=pydeck.map_styles.LIGHT_NO_LABELS,
    tooltip={
        "html": """
            <div style="font-family: 'Helvetica Neue', Arial, sans-serif; padding: 10px;">
                <b style="font-size: 1.2em;">H3 Cell:</b> <code>{h3_cell}</code><br/>
                <hr style="margin: 5px 0; border: 0; border-top: 1px solid #ccc;">
                <b>Records:</b> {n_records}<br/>
                <b>Datasets:</b> {datasets}
            </div>
        """,
        "style": {
            "width": "33%",
            "backgroundColor": "#2b2b2b",
            "color": "white",
            "borderRadius": "4px",
            "zIndex": "1000"
        }
    }
)
deck.to_html(f"h3_aggregated_resolution_{H3_RESOLUTION}.html")
deck.show()

### Plotting the Occurrences Over Time of a Particular Region

In [ ]:
occurences_over_time_chart = (
    seabird_atlas_sea_df
    .group_by(
        polars.col("eventDate").dt.year().alias("year"),
        polars.col("eventDate").dt.month().alias("month"),
    )
    .agg(
        polars.col("id").count().alias("occurrences"),
    )
    .plot.bar(
        x="year",
        y="occurrences",
        color="month",
    )
)
occurences_over_time_chart

### Identifying the Most Occurring Species

In [ ]:
top_ten_species_df = seabird_atlas_sea_df["scientificName"].value_counts(sort=True).head(10)
top_ten_species_list = top_ten_species_df["scientificName"].to_list()
display(top_ten_species_df)

### Plotting the Species Occurrences Over Time (Year)

In [ ]:
year_chart = (
    seabird_atlas_sea_df
    .filter(
        polars.col("scientificName").is_in(top_ten_species_list),
    )
    .group_by(
        polars.col("eventDate").dt.year().alias("year"),
        polars.col("scientificName"),
    )
    .agg(
        polars.col("id").count().alias("observations"),
    )
    .plot.bar(
        x="year",
        y="observations",
        color="scientificName",
    )
)
year_chart

### Plotting the Species Occurrences Over Time (Month)

In [ ]:
year = 1985
month_chart = (
    seabird_atlas_sea_df
    .filter(
        polars.col("eventDate").dt.year().eq(year),
        polars.col("scientificName").is_in(top_ten_species_list),
    )
    .group_by(
        polars.col("eventDate").dt.month().alias("month"),
        polars.col("scientificName"),
    )
    .agg(
        polars.col("id").count().alias("observations"),
    )
    .plot.bar(
        x="month",
        y="observations",
        color="scientificName",
    )
)
month_chart

### Plotting the Occurrences of a Particular Species Over Time

In [ ]:
year = 1985
puffin_chart = (
    seabird_atlas_sea_df
    .filter(
        polars.col("scientificName").eq("Puffinus tenuirostris"),
    )
    .group_by(
        polars.col("eventDate").dt.year().alias("year"),
        polars.col("eventDate").dt.month().alias("month"),
    )
    .agg(
        polars.col("id").count().alias("observations"),
    )
    .plot.bar(
        x="year",
        y="observations",
        color="month",
    )
)
puffin_chart